In [1]:
#CELL 1
# List of every Bronze table we have created so far
tables = ["bronze_orders", "bronze_customers", "bronze_order_items", "bronze_order_payments",
          "bronze_order_reviews", "bronze_products", "bronze_sellers", "bronze_geolocation",
          "bronze_category_translation", "bronze_warehouses", "bronze_shipments", "bronze_stock_depletions"]

# Loop over each table name
for t in tables:
    # Print the table name and how many rows it has
    print(t, "->", spark.table(t).count())

StatementMeta(, 8ab2387b-328e-4912-9546-75712e9dcce4, 3, Finished, Available, Finished, False)

bronze_orders -> 100041
bronze_customers -> 99441
bronze_order_items -> 112650
bronze_order_payments -> 103886
bronze_order_reviews -> 99224
bronze_products -> 32951
bronze_sellers -> 3095
bronze_geolocation -> 1000163
bronze_category_translation -> 71
bronze_warehouses -> 6
bronze_shipments -> 600
bronze_stock_depletions -> 600


In [2]:
#CELL 2
from pyspark.sql import functions as F    # Spark helper functions

# Function: counts missing values in every column of a table
def null_report(table):
    df = spark.table(table)                                   # load the table
    total = df.count()                                        # total number of rows
    # For each column, count the rows where the value is null
    nulls = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).collect()[0].asDict()
    print(f"\n{table} ({total} rows)")                        # print the table name and size
    for col_name, n in nulls.items():                         # loop over each column's result
        if n > 0:                                             # only show columns that have missing values
            print(f"  {col_name}: {n} nulls ({n/total:.1%})") # show count and percentage

# Run the check on the main tables
for t in ["bronze_orders", "bronze_products", "bronze_order_reviews", "bronze_customers"]:
    null_report(t)

StatementMeta(, 8ab2387b-328e-4912-9546-75712e9dcce4, 4, Finished, Available, Finished, False)


bronze_orders (100041 rows)
  order_approved_at: 160 nulls (0.2%)
  order_delivered_carrier_date: 1783 nulls (1.8%)
  order_delivered_customer_date: 2965 nulls (3.0%)

bronze_products (32951 rows)
  product_category_name: 610 nulls (1.9%)
  product_name_lenght: 610 nulls (1.9%)
  product_description_lenght: 610 nulls (1.9%)
  product_photos_qty: 610 nulls (1.9%)
  product_weight_g: 2 nulls (0.0%)
  product_length_cm: 2 nulls (0.0%)
  product_height_cm: 2 nulls (0.0%)
  product_width_cm: 2 nulls (0.0%)

bronze_order_reviews (99224 rows)
  review_comment_title: 87656 nulls (88.3%)
  review_comment_message: 58247 nulls (58.7%)

bronze_customers (99441 rows)


In [3]:
#CELL 3
# Function: compares total rows with unique key values
def dup_report(table, key):
    df = spark.table(table)                         # load the table
    total = df.count()                              # total rows
    unique = df.select(key).distinct().count()      # number of unique key values
    # If total > unique, some keys appear more than once
    print(f"{table}.{key}: {total} rows, {unique} unique, {total - unique} duplicates")

dup_report("bronze_orders", "order_id")             # each order should appear once
dup_report("bronze_customers", "customer_id")       # each customer_id should appear once
dup_report("bronze_products", "product_id")         # each product should appear once
dup_report("bronze_order_reviews", "review_id")     # known to have repeats in Olist
dup_report("bronze_shipments", "order_id")          # each order should have one shipment

StatementMeta(, 8ab2387b-328e-4912-9546-75712e9dcce4, 5, Finished, Available, Finished, False)

bronze_orders.order_id: 100041 rows, 100041 unique, 0 duplicates
bronze_customers.customer_id: 99441 rows, 99441 unique, 0 duplicates
bronze_products.product_id: 32951 rows, 32951 unique, 0 duplicates
bronze_order_reviews.review_id: 99224 rows, 98410 unique, 814 duplicates
bronze_shipments.order_id: 600 rows, 600 unique, 0 duplicates


In [4]:
#CELL 4
# Function: finds rows in table A whose ID does not exist in table B
def orphan_check(a, b, key, label):
    # left_anti join keeps only the rows of A with NO match in B
    n = spark.table(a).join(spark.table(b), key, "left_anti").count()
    print(f"{label}: {n} orphans")                  # 0 means everything links correctly

orphan_check("bronze_orders", "bronze_customers", "customer_id", "orders without a customer")
orphan_check("bronze_order_items", "bronze_orders", "order_id", "order items without an order")
orphan_check("bronze_order_items", "bronze_products", "product_id", "order items without a product")
orphan_check("bronze_shipments", "bronze_orders", "order_id", "shipments without an order")
orphan_check("bronze_shipments", "bronze_warehouses", "warehouse_id", "shipments without a warehouse")

StatementMeta(, 8ab2387b-328e-4912-9546-75712e9dcce4, 6, Finished, Available, Finished, False)

orders without a customer: 0 orphans
order items without an order: 0 orphans
order items without a product: 0 orphans
shipments without an order: 0 orphans
shipments without a warehouse: 0 orphans


In [5]:
#CELL 5
spark.table("bronze_orders").printSchema()      # check the timestamp columns show as "timestamp", not "string"
spark.table("bronze_shipments").printSchema()   # check the shipped_at and delivered_at types

StatementMeta(, 8ab2387b-328e-4912-9546-75712e9dcce4, 7, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- shipment_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- shipped_at: timestamp (nullable = true)
 |-- delivered_at: timestamp (nullable = true)
 |-- carrier: string (nullable = true)

